## Task 5: Content-Based TED Talks Recommendation Engine

### Objective
Develop a reusable recommendation function that recommends the five most similar TED Talks based on textual content using TF-IDF and cosine similarity.

In [51]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load cleaned dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Display basic information
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

df.head()

Dataset shape: (4005, 26)

Available columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text', 'combined_text_length', 'combined_text_word_count']


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text,combined_text_length,combined_text_word_count
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,with the same humor and humanity he exuded in ...,thank you so much chris and it s truly a great...,alternative energy cars climate change culture...,averting the climate crisis alternative energy...,11819,2269
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you ve ever seen,you ve never seen data presented like this wit...,about 10 years ago i took on the task to teach...,africa asia google demo economics global issue...,the best stats you ve ever seen africa asia go...,17185,3278
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...,computers entertainment interface design media...,simplicity sells computers entertainment inter...,17892,3548
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,in an emotionally charged talk macarthur winni...,if you re here today and i m very happy that y...,macarthur grant activism business cities envir...,greening the ghetto macarthur grant activism b...,18150,3226
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity,sir ken robinson makes an entertaining and pro...,good morning how are you audience good it s be...,children creativity culture dance education pa...,do schools kill creativity children creativity...,17061,3326


### 2. TF-IDF Vectorization

TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numerical values based on the importance of words in each TED Talk.

We use the `combined_text` column because it contains the combined textual content of the talks.

In [52]:
# Check for missing text
print("Missing combined_text values:", df["combined_text"].isnull().sum())

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

# Generate TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df["combined_text"])

# Display results
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Vocabulary Size:", len(tfidf.get_feature_names_out()))

# Display sample vocabulary
print("\nSample vocabulary:")
print(tfidf.get_feature_names_out()[:20])

Missing combined_text values: 0
TF-IDF Matrix Shape: (4005, 5000)
Vocabulary Size: 5000

Sample vocabulary:
['000' '10' '100' '11' '12' '120' '13' '14' '15' '150' '16' '17' '17th'
 '18' '18th' '19' '1950s' '1960s' '1970s' '1980s']


* TfidfVectorizer() initializes the vectorizer.

* max_features=5000 limits the vocabulary to 5,000 features.

* stop_words="english" removes common English words.

* fit_transform() learns the vocabulary and transforms the text into numerical vectors.

### 3. Cosine Similarity Analysis

Cosine similarity measures the similarity between two TF-IDF vectors.

It produces a similarity score:
- 1 means highly similar.
- 0 means no similarity in their vector representations.

We calculate the similarity matrix once and reuse it for generating recommendations.

In [53]:
# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(tfidf_matrix)

# Display matrix information
print("Cosine Similarity Matrix Shape:", similarity_matrix.shape)

# Display first 5 rows and columns
print("\nSample Similarity Matrix:")
print(similarity_matrix[:5, :5])

Cosine Similarity Matrix Shape: (4005, 4005)

Sample Similarity Matrix:
[[1.         0.15094786 0.24395183 0.17692886 0.24616646]
 [0.15094786 1.         0.15582193 0.15698043 0.13657543]
 [0.24395183 0.15582193 1.         0.13851393 0.25810331]
 [0.17692886 0.15698043 0.13851393 1.         0.13463015]
 [0.24616646 0.13657543 0.25810331 0.13463015 1.        ]]


### 4. Building the Recommendation Function

The `recommend_talks()` function accepts a TED Talk title and returns the most similar TED Talks.

The function:
1. Searches for the given TED Talk title.
2. Finds its index in the dataset.
3. Retrieves similarity scores from the precomputed cosine similarity matrix.
4. Excludes the selected TED Talk itself.
5. Sorts the remaining talks by similarity score.
6. Returns the top N recommendations in a DataFrame.

The TF-IDF vectors and cosine similarity matrix are calculated only once, outside this function.

In [54]:
def recommend_talks(title, top_n=5):

    # Validate top_n
    if not isinstance(top_n, int) or isinstance(top_n, bool):
        print("Error: top_n must be a positive integer.")
        return pd.DataFrame()

    if top_n <= 0:
        print("Error: top_n must be greater than zero.")
        return pd.DataFrame()

    # Check whether the title exists in the dataset
    if title not in df["title"].values:
        print(f"TED Talk '{title}' not found in the dataset.")
        return pd.DataFrame()

    # Get the index of the selected TED Talk
    talk_index = df[df["title"] == title].index[0]

    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[talk_index]))

    # Sort by similarity score in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude the selected TED Talk itself
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != talk_index
    ]

    # Select top N recommendations
    top_recommendations = similarity_scores[:top_n]

    # Create recommendation DataFrame
    recommendations = pd.DataFrame([
        {
            "TED Talk Title": df.iloc[index]["title"],
            "Similarity Score": round(score, 4)
        }
        for index, score in top_recommendations
    ])

    return recommendations

* Important lines:
- df["title"] == title: Finds the requested talk.
- similarity_matrix[talk_index]: Retrieves its similarity scores against all talks.
- item[0] != talk_index: Excludes the selected talk itself.
- top_recommendations: Keeps only the highest-scoring recommendations.

In [55]:
recommend_talks("Averting the climate crisis")

,TED Talk Title,Similarity Score
0,New thinking on the climate crisis,0.3676
1,How we can turn the tide on climate,0.3542
2,A climate solution where all sides can win,0.3390
3,A one-man world summit,0.3388
4,How we could change the planet's climate future,0.3383


In [56]:
recommend_talks("Averting the climate crisis", top_n=3)

,TED Talk Title,Similarity Score
0,New thinking on the climate crisis,0.3676
1,How we can turn the tide on climate,0.3542
2,A climate solution where all sides can win,0.3390


### 5. Testing the Recommendation Function

The recommendation function is tested using different TED Talk titles to evaluate whether it produces relevant recommendations.

In [57]:
# Test Case 1: Climate-related TED Talk
recommend_talks("Averting the climate crisis")

,TED Talk Title,Similarity Score
0,New thinking on the climate crisis,0.3676
1,How we can turn the tide on climate,0.3542
2,A climate solution where all sides can win,0.3390
3,A one-man world summit,0.3388
4,How we could change the planet's climate future,0.3383


In [58]:
# Test Case 2: Statistics-related TED Talk
recommend_talks("The best stats you've ever seen")

,TED Talk Title,Similarity Score
0,Let my dataset change your mindset,0.6495
1,New insights on poverty,0.4742
2,The good news of the decade? We're winning the...,0.4094
3,"Insights on HIV, in stunning data visuals",0.3860
4,"Global population growth, box by box",0.3775


In [59]:
# Test Case 3: Design-related TED Talk
recommend_talks("Simplicity sells")

,TED Talk Title,Similarity Score
0,Nerdcore comedy,0.3827
1,Design and discovery,0.3489
2,A comic sendup of TED2006,0.3326
3,Cool tricks your phone can do,0.3324
4,Designing for simplicity,0.3232


### 6. Handling Invalid TED Talk Titles

The function checks whether the entered title exists in the dataset. If the title is invalid, it displays an appropriate message and returns an empty DataFrame.

In [60]:
# Test Zero

recommend_talks("Averting the climate crisis", top_n=0)

Error: top_n must be greater than zero.


""


In [61]:
# Text instead of a number 

recommend_talks("Averting the climate crisis", top_n="five")

Error: top_n must be a positive integer.


""


In [62]:
# Test Negative number

recommend_talks("Averting the climate crisis", top_n=-2)

Error: top_n must be greater than zero.


""


In [63]:
# Test an invalid TED Talk title
invalid_result = recommend_talks("This TED Talk Does Not Exist")

display(invalid_result)

TED Talk 'This TED Talk Does Not Exist' not found in the dataset.


""


### 7. Observations

1. The recommendation engine successfully recommends five TED Talks based on textual similarity.
2. Climate-related talks generally receive recommendations related to climate change and environmental issues.
3. Statistics-related talks receive recommendations related to data, statistics, and visualization.
4. Design-related talks receive recommendations associated with creativity, design, and simplicity.
5. The selected TED Talk is excluded from its own recommendations.
6. Invalid TED Talk titles are handled appropriately.
7. The precomputed TF-IDF vectors and cosine similarity matrix make the recommendation function reusable without repeating the vectorization and similarity calculations.

### 8. Limitations

1. TF-IDF focuses on word frequency and importance rather than understanding the complete meaning of text.
2. Talks discussing similar topics with different vocabulary may not always be recommended.
3. Recommendations depend on the quality and availability of textual content.
4. The system does not consider user preferences, ratings, or viewing history.
5. Similarity scores do not guarantee that a user will enjoy a recommended talk.

### 9. Review Questions and Answers

#### Q1. What is a content-based recommendation system?
A content-based recommendation system recommends items by comparing their features or content with the content of an item selected by the user.

#### Q2. Why is TF-IDF used?
TF-IDF converts textual data into numerical vectors and assigns importance to words based on their frequency and rarity across documents.

#### Q3. What is cosine similarity?
Cosine similarity measures the similarity between two vectors by calculating the cosine of the angle between them.

#### Q4. Why do we exclude the selected TED Talk from recommendations?
We exclude it because a TED Talk should not recommend itself.

#### Q5. Why is the recommendation function reusable?
The TF-IDF matrix and cosine similarity matrix are calculated once outside the function. The function reuses these precomputed results for different user selections.

#### Q6. How does the system handle invalid titles?
It checks whether the entered title exists in the dataset. If not, it displays an error message and returns an empty DataFrame.

#### Q7. What are the limitations of this recommendation system?
It relies on textual similarity and does not understand context perfectly or consider user preferences and viewing history.